In [ ]:
pip install pyswip

In [ ]:
pip install flask


In [ ]:
from flask import Flask, request, jsonify
from pyswip import Prolog

app = Flask(__name__)
prolog = Prolog()
prolog.consult("rules.pl")

@app.route('/inheritance-calculator', methods=['POST'])
def inheritance_calculator():
    try:
        # Get JSON data from the request
        data = request.get_json()
        total_wealth = data['total_wealth']
        has_husband = data.get('has_husband', 0)
        num_wives = data.get('num_wives', 0)
        num_sons = data.get('num_sons', 0)
        num_daughters = data.get('num_daughters' ,0)
        num_sons_sons = data.get('num_sons_sons' ,0)
        num_sons_daughters = data.get('num_sons_daughters' ,0)

        # Validate input constraints
        if num_wives > 4:
            return jsonify({"error": "Number of wives cannot exceed 4"}), 400
        if has_husband == 1 and num_wives != 0:
            return jsonify({"error": "If husband is present, number of wives must be 0"}), 400

        # Query the Prolog engine
        result = list(prolog.query(
            f"inheritance_calculator({total_wealth}, {has_husband}, {num_wives}, {num_sons}, {num_daughters},{num_sons_sons} ,{num_sons_daughters},HusbandShare, PerWifeShare, PerSonShare, PerDaughterShare, PerSonsDaughterShare)"
        ))

        # Process the result
        if result:
            response = result[0]
            return jsonify({
                "husband_share": response['HusbandShare'],
                "per_wife_share": response['PerWifeShare'],
                "per_son_share": response['PerSonShare'],
                "per_daughter_share": response['PerDaughterShare'],
                "per_sons_daughter_share": response['PerSonsDaughterShare']
            })
        else:
            return jsonify({"error": "Calculation failed"}), 400
    except KeyError as e:
        return jsonify({"error": f"Missing required field: {str(e)}"}), 400
    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == '__main__':
    app.run(debug=False)

In [1]:
from pyswip import Prolog


prolog = Prolog()
prolog.consult("rules.pl")


def inheritance_calculator():
    total_wealth = 1000
    has_husband = 1
    num_wives = 0
    num_sons = 1
    num_daughters = 1
    num_sons_sons = 1
    num_sons_daughters = 1

    # Query the Prolog engine
    result = list(prolog.query(
        f"inheritance_calculator({total_wealth}, {has_husband}, {num_wives}, {num_sons}, {num_daughters},{num_sons_sons} ,{num_sons_daughters},HusbandShare, PerWifeShare, PerSonShare, PerDaughterShare, PerSonsDaughterShare)"
    ))

    if result:
        response = result[0]
        print("Husband Share:", response['HusbandShare'])
        print("Per Wife Share:", response['PerWifeShare'])
        print("Per Son Share:", response['PerSonShare'])
        print("Per Daughter Share:", response['PerDaughterShare'])
        print("Per Son's Daughter Share:", response['PerSonsDaughterShare'])




inheritance_calculator()

Husband Share: 250
Per Wife Share: 0
Per Son Share: 500
Per Daughter Share: 250
Per Son's Daughter Share: 0
